In [ ]:
# Surface Roughness Prediction in 3D Printing

In [ ]:
# TEAM MEMBERS
# 1. Aarushi Jain - 00601042024 (MAE)
# 2. Dishika Singh- 02501042024 (MAE)

In [ ]:
# Project type: Supervised Machine Learning – Regression
# Dataset: 3D Printer Dataset for Mechanical Engineers (50 observations)

## Objective - Predict the surface roughness of an FDM-printed part from its printing parameters. Surface roughness is the target variable, measured in micrometres (µm).

## Input features
# Layer height
# Wall thickness
# Infill density
# Infill pattern
# Nozzle temperature
# Bed temperature
# Print speed
# Material
# Fan speed

## Machine Learning models

#Linear Regression, Decision Tree Regressor, Random Forest Regressor and Support Vector Regression (SVR).
#Important limitation: The dataset contains only 50 observations. Results are useful for a small academic project, but they should not be treated as proof that the model will generalise to every printer, material or operating condition.


In [ ]:
# IMPORT LIBRARIES

import warnings                                         # To handle warning messages
warnings.filterwarnings("ignore")                       # To hide unnecessary warnings

import numpy as np                                      # For numerical operations
import pandas as pd                                     # For data handling and analysis
import matplotlib.pyplot as plt                         # For creating graphs and plots
import seaborn as sns                                   # For statistical data visualisation

from sklearn.model_selection import train_test_split, KFold, cross_validate    # For splitting data and cross-validation
from sklearn.compose import ColumnTransformer                                  # To apply different preprocessing to different columns
from sklearn.preprocessing import OneHotEncoder, StandardScaler                # To encode categorical data and scale numerical data
from sklearn.pipeline import Pipeline                                          # To combine preprocessing and ML model steps
from sklearn.impute import SimpleImputer                                       # To handle missing values
from sklearn.linear_model import LinearRegression                              # Linear Regression model
from sklearn.tree import DecisionTreeRegressor                                 # Decision Tree regression model
from sklearn.ensemble import RandomForestRegressor                             # Random Forest regression model
from sklearn.svm import SVR                                                    # Support Vector Regression model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # For evaluating model performance
from sklearn.inspection import permutation_importance                          # To identify important input features

RANDOM_STATE = 42  # Fixed value for reproducible results
sns.set_theme(style="whitegrid")  # Sets graph style with a white background and grid


In [ ]:
#unzip zip file to csv
!unzip /content/data.zip -d /content/extracted_folder

unzip:  cannot find or open /content/data.zip, /content/data.zip.zip or /content/data.zip.ZIP.


In [ ]:
# DATASET

try:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
except ImportError:
    file_name = "data.csv"

df = pd.read_csv(file_name)
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nDataset information:")
df.info()

print("\\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))

print("Duplicate rows:", df.duplicated().sum())
print("\\nStatistical summary:")
display(df.describe(include="all").T)

In [ ]:
# EXPLORATORY DATA ANALYSIS (EDA) : These help us understand the target variable and explore how the printing parameters vary. They show patterns in this dataset; they do not by themselves establish causation.

# Distribution of surface roughness
plt.figure(figsize=(7, 4))
sns.histplot(df["roughness"], kde=True)  # Shows frequency and distribution
plt.title("Distribution of Surface Roughness")
plt.xlabel("Roughness (µm)")
plt.tight_layout()
plt.show()

# Detecting outliers in surface roughness
plt.figure(figsize=(7, 4))
sns.boxplot(x=df["roughness"])
plt.title("Surface Roughness – Box Plot")
plt.xlabel("Roughness (µm)")
plt.tight_layout()
plt.show()

# Correlation between numerical features
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numerical Features")
plt.tight_layout()
plt.show()

# Comparing roughness across categorical features
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=df, x="infill_pattern", y="roughness", ax=axes[0])
axes[0].set_title("Roughness by Infill Pattern")

sns.boxplot(data=df, x="material", y="roughness", ax=axes[1])
axes[1].set_title("Roughness by Material")

plt.tight_layout()
plt.show()

In [ ]:
# DEFINING INPUT FEATURES AND TARGET
target = "roughness"  # Column to be predicted

X = df.drop(columns=[target, "tension_strenght", "elongation"])  # Input features (all columns except target, tension_strenght, elongation)
y = df[target]  # Target variable (surface roughness)

# Separating categorical and numerical features
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=np.number).columns.tolist()

# Displaying the identified features
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Target:", target)

In [ ]:
# DATA SPLITTING : We keep 20% of the observations as a test set. The test set is kept separate from model selection and cross-validation.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


In [ ]:
# COMMON PREPROCESSING :

# Handling missing values
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

for col in numerical_features:
    median = X_train_clean[col].median()
    X_train_clean[col] = X_train_clean[col].fillna(median)
    X_test_clean[col] = X_test_clean[col].fillna(median)

for col in categorical_features:
    mode = X_train_clean[col].mode()[0]
    X_train_clean[col] = X_train_clean[col].fillna(mode)
    X_test_clean[col] = X_test_clean[col].fillna(mode)

# One-hot encoding of categorical features
X_train_encoded = pd.get_dummies(X_train_clean, columns=categorical_features, dtype=int)
X_test_encoded = pd.get_dummies(X_test_clean, columns=categorical_features, dtype=int)

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

# Converting data into NumPy arrays
X_train_np = X_train_encoded.to_numpy(dtype=float)
X_test_np = X_test_encoded.to_numpy(dtype=float)

y_train_np = y_train.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float)

# Standardising input features
mean = X_train_np.mean(axis=0)
std = X_train_np.std(axis=0)
std[std == 0] = 1

X_train_scaled = (X_train_np - mean) / std
X_test_scaled = (X_test_np - mean) / std



In [ ]:
# Linear Regression
# Initialising weights and bias
weights = np.zeros(X_train_scaled.shape[1])
bias = 0

learning_rate = 0.01
epochs = 5000
n = len(y_train_np)

# Training using Gradient Descent
for i in range(epochs):

    y_pred = np.dot(X_train_scaled, weights) + bias
    error = y_pred - y_train_np

    dw = (2 / n) * np.dot(X_train_scaled.T, error)
    db = (2 / n) * np.sum(error)

    weights -= learning_rate * dw
    bias -= learning_rate * db

# Predicting on test data
linear_pred = np.dot(X_test_scaled, weights) + bias

In [ ]:
# Decision Tree
# Building a Decision Tree
def build_tree(X, y, depth=0, max_depth=4, min_samples=2):

    if depth >= max_depth or len(y) < min_samples:
        return np.mean(y)

    best_error = float("inf")
    best_split = None

    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])

        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:

            left = X[:, feature] <= threshold
            right = ~left

            if left.sum() == 0 or right.sum() == 0:
                continue

            error = (
                np.sum((y[left] - np.mean(y[left])) ** 2) +
                np.sum((y[right] - np.mean(y[right])) ** 2)
            )

            if error < best_error:
                best_error = error
                best_split = (feature, threshold)

    if best_split is None:
        return np.mean(y)

    feature, threshold = best_split

    left = X[:, feature] <= threshold
    right = ~left

    return {
        "feature": feature,
        "threshold": threshold,
        "left": build_tree(X[left], y[left], depth + 1, max_depth, min_samples),
        "right": build_tree(X[right], y[right], depth + 1, max_depth, min_samples)
    }


# Making predictions using the tree
def predict_tree(tree, row):

    if not isinstance(tree, dict):
        return tree

    if row[tree["feature"]] <= tree["threshold"]:
        return predict_tree(tree["left"], row)

    return predict_tree(tree["right"], row)


# Training the Decision Tree
decision_tree = build_tree(
    X_train_np, y_train_np, max_depth=4, min_samples=2
)

# Predicting on test data
dt_pred = np.array([
    predict_tree(decision_tree, row) for row in X_test_np
])

In [ ]:
# Random Forest
# Building a Random Forest tree
def build_rf_tree(X, y, depth=0, max_depth=4, min_samples=2):

    if depth >= max_depth or len(y) < min_samples:
        return np.mean(y)

    features = np.random.choice(
        X.shape[1],
        max(1, int(np.sqrt(X.shape[1]))),
        replace=False
    )

    best_error = float("inf")
    best_split = None

    for feature in features:
        values = np.unique(X[:, feature])
        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:

            left = X[:, feature] <= threshold
            right = ~left

            if left.sum() == 0 or right.sum() == 0:
                continue

            error = (
                np.sum((y[left] - np.mean(y[left])) ** 2) +
                np.sum((y[right] - np.mean(y[right])) ** 2)
            )

            if error < best_error:
                best_error = error
                best_split = (feature, threshold)

    if best_split is None:
        return np.mean(y)

    feature, threshold = best_split

    left = X[:, feature] <= threshold
    right = ~left

    return {
        "feature": feature,
        "threshold": threshold,
        "left": build_rf_tree(X[left], y[left], depth + 1, max_depth, min_samples),
        "right": build_rf_tree(X[right], y[right], depth + 1, max_depth, min_samples)
    }


# Creating multiple trees using bootstrap sampling
np.random.seed(RANDOM_STATE)

forest = []
n_trees = 100

for i in range(n_trees):

    indices = np.random.choice(
        len(X_train_np), len(X_train_np), replace=True
    )

    tree = build_rf_tree(
        X_train_np[indices],
        y_train_np[indices],
        max_depth=4,
        min_samples=2
    )

    forest.append(tree)

# Predicting using a single tree
def predict_tree(tree, row):

    if not isinstance(tree, dict):
        return tree

    feature = tree["feature"]
    threshold = tree["threshold"]

    if row[feature] <= threshold:
        return predict_tree(tree["left"], row)
    else:
        return predict_tree(tree["right"], row)

# Predicting using all trees
rf_pred = np.array([
    np.mean([predict_tree(tree, row) for tree in forest])
    for row in X_test_np
])

In [ ]:
# Support Vector Regression (SVR)
# Initialising SVR parameters
weights = np.zeros(X_train_scaled.shape[1])
bias = 0

learning_rate = 0.005
C = 10
epsilon = 0.1
epochs = 5000
n = len(y_train_np)

# Training SVR using subgradient descent
for i in range(epochs):

    y_pred = np.dot(X_train_scaled, weights) + bias
    error = y_train_np - y_pred

    outside_margin = np.abs(error) > epsilon

    gradient_w = weights - (
        C / n
    ) * np.dot(
        X_train_scaled.T,
        np.sign(error) * outside_margin
    )

    gradient_b = -(
        C / n
    ) * np.sum(np.sign(error) * outside_margin)

    weights -= learning_rate * gradient_w
    bias -= learning_rate * gradient_b

# Predicting on test data
svr_pred = np.dot(X_test_scaled, weights) + bias

In [ ]:
# ALL MODELS

predictions = {
    "Linear Regression": linear_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "SVR": svr_pred
}

In [ ]:
# Calculating evaluation metrics
def evaluate_model(y_true, y_pred):

    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)

    ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_residual = np.sum((y_true - y_pred) ** 2)

    r2 = 1 - (ss_residual / ss_total)

    return mae, rmse, r2

In [ ]:
# Compare all four models
# Evaluating all four models
results = []

predictions = {
    "Linear Regression": linear_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "SVR": svr_pred
}

for name, pred in predictions.items():

    mae, rmse, r2 = evaluate_model(y_test_np, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("MAE")

display(results_df)

In [ ]:
# Comparing actual and predicted roughness
plt.figure(figsize=(8, 5))

model_predictions = {
    "Linear Regression": (linear_pred, "blue"),
    "Decision Tree": (dt_pred, "orange"),
    "Random Forest": (rf_pred, "green"),
    "SVR": (svr_pred, "red")
}

for name, (pred, color) in model_predictions.items():

    # Sorting points according to actual values
    sorted_indices = np.argsort(y_test_np)

    actual_sorted = y_test_np[sorted_indices]
    pred_sorted = pred[sorted_indices]

    # Connecting points with a transparent dotted line
    plt.plot(
        actual_sorted, pred_sorted,
        color=color, linestyle=":",
        alpha=0.45, linewidth=1.5
    )

    # Plotting individual points
    plt.scatter(
        y_test_np, pred,
        color=color, label=name, alpha=0.85
    )

# Perfect prediction reference line
min_val = min(y_test_np.min(), min(p.min() for p in predictions.values()))
max_val = max(y_test_np.max(), max(p.max() for p in predictions.values()))

plt.plot([min_val, max_val], [min_val, max_val], 'k--')

plt.xlabel("Actual Roughness (µm)")
plt.ylabel("Predicted Roughness (µm)")
plt.title("Actual vs Predicted Surface Roughness")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Comparing all models
plt.figure(figsize=(8, 5))

sns.barplot(data=results_df, x="Model", y="MAE")

plt.title("Model Performance Comparison")
plt.xlabel("Machine Learning Models")
plt.ylabel("Mean Absolute Error")
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Selecting the best model based on MAE
best_model = results_df.iloc[0]["Model"]

print("Best Performing Model:", best_model)
print("MAE:", round(results_df.iloc[0]["MAE"], 2))
print("RMSE:", round(results_df.iloc[0]["RMSE"], 2))
print("R2 Score:", round(results_df.iloc[0]["R2 Score"], 2))